In [1]:
import pandas as pd
import requests
import json
import glob
import os
import urllib3
import httpx
import time
import shutil

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## GET AUTHORISATION TOKEN
Grab your token and your client ID used to request an authorisation token (https://supercoach.heraldsun.com.au/2020/api/afl/classic/v1/access_token) and input them into the payload

In [2]:
# Start the session
session = requests.Session()
# access_token = "eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6Ik1UVkdNRGxETUVaRk1qWkdOVU0yT0RnNFF6VTROalZCTlRsR1FrUTRSamM1TVROQ09UWTJOZyJ9.eyJodHRwOi8vbG9naW4ubmV3c2NvcnBhdXN0cmFsaWEuY29tLmF1L3Byb2ZpbGUiOnsiYXRzX2hhc2giOiJiNGZlMmY5OTQxYWY0YWY0YjljZDE4ZjYwOWQ1MDIxNWU4ZjBkNGRkYjkzODUxMmUwODhkNmVkMWRjMDYzMTQ0LjI3ZTcxZjBkOWIwNzFhZWMyN2QxNTdmNjNiNzg1OWJhOTlhYjRmOWYiLCJhdXRoUHJvdmlkZXIiOiJhdXRoMCIsImN1c3RvbWVyX3Byb2R1Y3RfaG9sZGluZyI6WyJaWl9OTF9SRUciLCJaWl9TQ19EREZSTk9FTlQiLCJaWl9DU19ERCJdLCJoYXNfY29uc2VudGVkX3RvX3RjIjp0cnVlLCJyYXQiOjE1NTQ4ODU5OTYsInNpdGUiOiJzdXBlcmNvYWNoX2hlcmFsZHN1biIsInRoaW5rX2lkIjoiMTQxMjMyMzEifSwibmlja25hbWUiOiJNYXJrIiwibmFtZSI6Ik1hcmsiLCJwaWN0dXJlIjoiaHR0cHM6Ly9zLmdyYXZhdGFyLmNvbS9hdmF0YXIvNzdmMzczZGM1MTViMTNmMGU2ODc1ODRiZTNlMWVkOWI_cz00ODAmcj1wZyZkPWh0dHBzJTNBJTJGJTJGY2RuLmF1dGgwLmNvbSUyRmF2YXRhcnMlMkZyaS5wbmciLCJ1cGRhdGVkX2F0IjoiMjAyNS0wOC0wMlQxMToyNjoxMi42NTRaIiwiaXNzIjoiaHR0cHM6Ly9sb2dpbi5uZXdzY29ycGF1c3RyYWxpYS5jb20vIiwiYXVkIjoiWllDb3RsaWhxYUd1YXFTc1N2dTBMMnZ4RGRRWEN3MTYiLCJzdWIiOiJhdXRoMHxjNWNlZWM5Yi1hZDg0LTQ1YTktYWU1Yy05MGI2NzVjNGIxYzEiLCJpYXQiOjE3NTQxMzM5NzMsImV4cCI6MTc2MTQ3Nzk3Mywic2lkIjoiTjhVU0tmWHJ4bExfNENrRHcyY0ZZLTNodnRGVlhOd24iLCJhdF9oYXNoIjoiU1ZrTDF5YXh1RmxISHpYYXhCVkxEZyIsIm5vbmNlIjoid2RfejJmQjV4WTgwLnFwMmlacUZ2Z0s3amdnU0hvSU0ifQ.Al2Ey_OmOK75hxFWxQotjMXlBuU2-yuK0gnAXv3_OfLrr08mYMF3_lDhu4KG-_LlNGRvmgkKKSgZWhVfduqXTt41xqchid4NmHmEWmxSgyeEUgI3VJWciOGeyAP9hGZ95pIukVNcCea9PSehyqxr-9sHSqtF-nuztYqol7pkkBtokxz2PBHC3zQR1TRQFzmlzlHk8kAcHLAiW2kOHgzNYHFQOutGqP8PkuK8gRayMqQjs2pPASjR9zaN85MZOoIMzoStfKfx7qSTvBdKsuBZqZdhx2hu_skx8ygEqqrTnUXASfB4bb_1KPY3zNGnev7GseVaUQdSm_FU4LzMqyx0fw"
access_token = "b849fa19223505f95e25c7ec130ebbb82c40fb10"
headers = {
    "Authorization": "Bearer " + access_token
}

## ENTER YEAR AND ROUND NUMBER

In [3]:
# specify year
year_number = 2025

# specify round
round_number = 21

## PLAYER LIST

In [4]:
def get_position_from_json(positions):
    positions_string = ''
    for position in positions:
        if len(positions_string) > 0:
            positions_string = positions_string + ' ' + position['position']
        else:
            positions_string = positions_string + position['position']
    return positions_string

In [ ]:
s = session.get("https://www.supercoach.com.au/2025/api/afl/draft/v1/players-cf?embed=notes%2Codds%2Cplayer_stats%2Cpositions&round={}".format(round_number), headers={**headers, 'User-Agent': 'Mozilla/5.0'}, verify=False)
if s.status_code == 200:
    player_json = json.loads(s.text)
else:
    raise Exception(f'Error fetching data: {s.status_code} - {s.text}')
df_player_list = pd.DataFrame()

for player in player_json:
    positions_string = get_position_from_json(player['positions'])
    df_line = pd.DataFrame(
        {
            'player_id': [player['id']],
            'feed_id': [player['feed_id']],
            'first_name': [player['first_name']],
            'last_name': [player['last_name']],
            'team': [player['team']['abbrev']],
            'position': [pos for pos in [player['positions']]],
            'player_id': [player['player_stats'][0]['player_id']],
            'round': [player['player_stats'][0]['round']],
            'points': [player['player_stats'][0]['points']],
            'price': [player['player_stats'][0]['price']],
            'total_points': [player['player_stats'][0]['total_points']],
            'position': [player['player_stats'][0]['position']],
            'games': [player['player_stats'][0]['games']],
            'total_games': [player['player_stats'][0]['total_games']],
            'price_change': [player['player_stats'][0]['price_change']],
            'total_price_change': [player['player_stats'][0]['total_price_change']],
            'last_position': [player['player_stats'][0]['last_position']],
            'round_position': [player['player_stats'][0]['round_position']],
            'minutes_played': [player['player_stats'][0]['minutes_played']],
            'total_minutes_played': [player['player_stats'][0]['total_minutes_played']],
            'updated_at': [player['player_stats'][0]['updated_at']],
            'kicks': [player['player_stats'][0]['kicks']],
            'total_kicks': [player['player_stats'][0]['total_kicks']],
            'handballs': [player['player_stats'][0]['handballs']],
            'total_handballs': [player['player_stats'][0]['total_handballs']],
            'marks': [player['player_stats'][0]['marks']],
            'total_marks': [player['player_stats'][0]['total_marks']],
            'tackles': [player['player_stats'][0]['tackles']],
            'total_tackles': [player['player_stats'][0]['total_tackles']],
            'freekicks_for': [player['player_stats'][0]['freekicks_for']],
            'total_freekicks_for': [player['player_stats'][0]['total_freekicks_for']],
            'freekicks_against': [player['player_stats'][0]['freekicks_against']],
            'total_freekicks_against': [player['player_stats'][0]['total_freekicks_against']],
            'hitouts': [player['player_stats'][0]['hitouts']],
            'total_hitouts': [player['player_stats'][0]['total_hitouts']],
            'goals': [player['player_stats'][0]['goals']],
            'total_goals': [player['player_stats'][0]['total_goals']],
            'behinds': [player['player_stats'][0]['behinds']],
            'total_behinds': [player['player_stats'][0]['total_behinds']],
            'avg5': [player['player_stats'][0]['avg5']],
            'own_raw': [player['player_stats'][0]['own_raw']],
            'avg': [player['player_stats'][0]['avg']],
            'avg3': [player['player_stats'][0]['avg3']],
            'opp': [player['player_stats'][0]['opp']['abbrev']],
            'oppavg': [player['player_stats'][0]['oppavg']],
            'opph': [player['player_stats'][0]['opph']],
            'opp1': [player['player_stats'][0]['opp1']],
            'opp1h': [player['player_stats'][0]['opp1h']],
            'opp2': [player['player_stats'][0]['opp2']],
            'opp2h': [player['player_stats'][0]['opp2h']],
            'opp3': [player['player_stats'][0]['opp3']],
            'opp3h': [player['player_stats'][0]['opp3h']],
            'ven': [player['player_stats'][0]['ven']['name']],
            'venavg': [player['player_stats'][0]['venavg']],
            'ven1': [player['player_stats'][0]['ven1']],
            'ven2': [player['player_stats'][0]['ven2']],
            'ven3': [player['player_stats'][0]['ven3']],
            # 'livepts': [player['player_stats'][0]['livepts']],
            # 'livegames': [player['player_stats'][0]['livegames']],
            'owned': [player['player_stats'][0]['owned']],
            'points_per_min': [player['player_stats'][0]['points_per_min']],
            'total_points_per_min': [player['player_stats'][0]['total_points_per_min']],
            'mvp_value': [player['player_stats'][0]['mvp_value']],
            'position_change': [player['player_stats'][0]['position_change']],
            # 'position_ranks': [player['player_stats'][0]['position_ranks']]
        }
    )
    df_player_list = pd.concat([df_player_list, df_line])
df_player_list.to_csv('outputs/player_list.csv')
    
df_player_list

C:\Users\Richa\AppData\Local\Temp\ipykernel_5600\2308298716.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_player_list = pd.concat([df_player_list, df_line])


,player_id,feed_id,first_name,last_name,team,position,round,points,price,total_points,...,ven,venavg,ven1,ven2,ven3,owned,points_per_min,total_points_per_min,mvp_value,position_change
0,695,297373,Marcus,Bontempelli,WBD,41,21,151,687700,1934,...,Docklands Stadium,113.0150,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",99.424700,1.525253,1.259115,5333.778009,14
0,441,290528,Max,Gawn,MEL,2,21,131,608500,2507,...,Docklands Stadium,109.5930,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",99.691800,1.169643,1.150000,4854.407659,-1
0,529,1006121,Zak,Butters,PTA,36,21,45,565700,1974,...,Kardinia Park,111.7500,"{'id': 1, 'name': 'Adelaide Oval', 'display_na...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...",99.198700,0.459184,1.096667,4871.768374,-5
0,75,293535,Lachie,Neale,BRL,21,21,142,536400,2119,...,Melbourne Cricket Ground,108.5600,"{'id': 4, 'name': 'Brisbane Cricket Ground', '...","{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 4, 'name': 'Brisbane Cricket Ground', '...",98.664500,1.314815,0.982383,5062.765455,4
0,518,1004965,Tristan,Xerri,NTH,27,21,0,573500,2065,...,Docklands Stadium,98.4194,"{'id': 11, 'name': 'Manuka Oval', 'display_nam...","{'id': 226, 'name': 'Ninja Stadium', 'display_...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",98.027500,NaN,1.130268,4721.291502,-8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,766,294859,Jeremy,*** McGovern,WCE,417,21,0,509700,535,...,Docklands Stadium,92.8462,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 53, 'name': 'Perth Stadium', 'display_n...",37.497400,NaN,0.930435,5716.259545,-4
0,7,297401,Matt,*** Crouch,ADE,424,21,0,460600,521,...,Adelaide Oval,97.4186,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",59.071300,NaN,1.037849,5304.416624,-6
0,101,295518,Sam,*** Docherty,CAR,217,21,0,429800,1219,...,Perth Stadium,97.8750,"{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",33.367600,NaN,0.799344,5288.759110,-17
0,774,296296,Dom,*** Sheed,WCE,652,21,0,238400,0,...,Docklands Stadium,68.5882,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 53, 'name': 'Perth Stadium', 'display_n...",0.636943,NaN,NaN,NaN,-8


: 

# Get player stats for every game

In [28]:
# Define your headers with authentication token
headers = {
    'User-Agent': 'Mozilla/5.0',
    'Authorization': 'Bearer ' + access_token  # Replace with your actual access token
}

# Create the directory if it doesn't exist
output_dir = 'inputs/player_stats_current'
os.makedirs(output_dir, exist_ok=True)

# Get the total number of players
total_players = len(df_player_list)
print(f"Total number of players: {total_players}")

# Iterate through each player in df_player_list using enumerate to get the correct index
for index, row in enumerate(df_player_list.itertuples(), start=1):
    player_id = row.player_id
    url = f"https://www.supercoach.com.au/2025/api/afl/draft/v1/players/{player_id}?embed=notes,odds,player_stats,player_match_stats,positions,trades"
    
    # Define the output file path
    output_file = os.path.join(output_dir, f'player_stats_current_season_{player_id}.json')
    
    # Check if the output file already exists
    if not os.path.exists(output_file):
        # Make the request to the URL
        response = httpx.get(url, headers=headers, verify=False)
        
        if response.status_code == 200:
            player_stats = response.json()
            # Save the response to a file
            with open(output_file, 'w') as f:
                json.dump(player_stats, f)
            
            # Calculate and print the completion percentage within the same print statement
            completion_percentage = (index / total_players) * 100
            print(f"Player ID: {player_id}, Stats saved to {output_file}, Completion: {completion_percentage:.2f}%")
        else:
            print(f"Error fetching data for Player ID: {player_id}, Status Code: {response.status_code}")
    else:
        print(f"Data for Player ID: {player_id} already exists in {output_file}")

Total number of players: 808
Data for Player ID: 695 already exists in inputs/player_stats_current\player_stats_current_season_695.json
Data for Player ID: 441 already exists in inputs/player_stats_current\player_stats_current_season_441.json
Data for Player ID: 529 already exists in inputs/player_stats_current\player_stats_current_season_529.json
Data for Player ID: 75 already exists in inputs/player_stats_current\player_stats_current_season_75.json
Data for Player ID: 518 already exists in inputs/player_stats_current\player_stats_current_season_518.json
Data for Player ID: 272 already exists in inputs/player_stats_current\player_stats_current_season_272.json
Data for Player ID: 509 already exists in inputs/player_stats_current\player_stats_current_season_509.json
Data for Player ID: 136 already exists in inputs/player_stats_current\player_stats_current_season_136.json
Data for Player ID: 671 already exists in inputs/player_stats_current\player_stats_current_season_671.json
Data for P

## extract relevant data from saved files above

In [26]:
# Define the directory containing the JSON files
input_dir = 'inputs/player_stats_current'

# Initialize a list to hold the extracted data
extracted_data = []

# Iterate through each file in the directory
for filename in os.listdir(input_dir):
    if filename.endswith('.json'):
        file_path = os.path.join(input_dir, filename)
        
        # Load the JSON data from the file
        with open(file_path, 'r') as file:
            data = json.load(file)
        
        # Extract the relevant fields into a list of dictionaries
        player_match_stats = data['player_match_stats']
        for match in player_match_stats:
            round_number = match['round']
            extracted_data.append({
                'feed_id': data['feed_id'],
                'player_id': match['player_id'],
                'first_name': data['first_name'],
                'last_name': data['last_name'],
                'team_abbrev': data['team']['abbrev'],
                'pos_1': data['positions'][0]['position'],
                'pos_2': data['positions'][1]['position'] if len(data['positions']) > 1 else None,
                'round': match['round'],
                'points': match['points'],
                'played': data['player_stats'][round_number]['total_games'],
                'avg': data['player_stats'][round_number]['avg'],
                'avg3': data['player_stats'][round_number]['avg3'],
                'avg5': data['player_stats'][round_number]['avg5'],
                'price': data['player_stats'][round_number]['price']
            })

# Create a DataFrame from the extracted data
df_player_stats_current = pd.DataFrame(extracted_data)

# Save the DataFrame to a CSV file
df_player_stats_current.to_csv('inputs/player_stats_current.csv', index=False)

# Display the DataFrame
display(df_player_stats_current)

,feed_id,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round,points,played,avg,avg3,avg5,price
0,1012807,1,Sam,Berry,ADE,MID,None,1,80,1,80.0000,80.0000,80.00,226900
1,1012807,1,Sam,Berry,ADE,MID,None,2,52,2,66.0000,66.0000,66.00,226900
2,1012807,1,Sam,Berry,ADE,MID,None,4,33,3,55.0000,55.0000,55.00,243100
3,1012807,1,Sam,Berry,ADE,MID,None,5,56,4,55.2500,47.0000,55.25,244400
4,1012807,1,Sam,Berry,ADE,MID,None,6,31,5,50.4000,40.0000,50.40,235500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9512,996731,99,Charlie,Curnow,CAR,FWD,None,18,89,17,85.6471,58.6667,74.80,394800
9513,996731,99,Charlie,Curnow,CAR,FWD,None,19,106,18,86.7778,76.0000,80.00,393700
9514,996731,99,Charlie,Curnow,CAR,FWD,None,20,119,19,88.4737,104.6670,80.20,431500
9515,996731,99,Charlie,Curnow,CAR,FWD,None,21,33,20,85.7000,86.0000,76.00,433900


# Get complete player stats pack for each player for past seasons

In [10]:
# Define your headers with authentication token
headers = {
    'User-Agent': 'Mozilla/5.0',
    'Authorization': 'Bearer ' + access_token  # Replace with your actual access token
}

# Create the directory if it doesn't exist
output_dir = 'inputs/player_stats_pack'
os.makedirs(output_dir, exist_ok=True)

# Get the total number of players
total_players = len(df_player_list)
print(f"Total number of players: {total_players}")

# Iterate through each player in df_player_list using enumerate to get the correct index
for index, row in enumerate(df_player_list.itertuples(), start=1):
    player_id = row.player_id
    url = f"https://www.supercoach.com.au/2024/api/afl/draft/v1/completeStatspack?player_id={player_id}"
    
    # Define the output file path
    output_file = os.path.join(output_dir, f'completestatspack_{player_id}.json')
    
    # Check if the output file already exists
    if not os.path.exists(output_file):
        # Make the request to the URL
        response = httpx.get(url, headers=headers, verify=False)
        
        if response.status_code == 200:
            player_stats = response.json()
            # Save the response to a file
            with open(output_file, 'w') as f:
                json.dump(player_stats, f)
            
            # Calculate and print the completion percentage within the same print statement
            completion_percentage = (index / total_players) * 100
            print(f"Player ID: {player_id}, Stats saved to {output_file}, Completion: {completion_percentage:.2f}%")
        else:
            print(f"Error fetching data for Player ID: {player_id}, Status Code: {response.status_code}")
    else:
        print(f"Data for Player ID: {player_id} already exists in {output_file}")

Total number of players: 808
Data for Player ID: 695 already exists in inputs/player_stats_pack\completestatspack_695.json
Data for Player ID: 441 already exists in inputs/player_stats_pack\completestatspack_441.json
Data for Player ID: 529 already exists in inputs/player_stats_pack\completestatspack_529.json
Data for Player ID: 75 already exists in inputs/player_stats_pack\completestatspack_75.json
Data for Player ID: 518 already exists in inputs/player_stats_pack\completestatspack_518.json
Data for Player ID: 272 already exists in inputs/player_stats_pack\completestatspack_272.json
Data for Player ID: 509 already exists in inputs/player_stats_pack\completestatspack_509.json
Data for Player ID: 136 already exists in inputs/player_stats_pack\completestatspack_136.json
Data for Player ID: 671 already exists in inputs/player_stats_pack\completestatspack_671.json
Data for Player ID: 245 already exists in inputs/player_stats_pack\completestatspack_245.json
Data for Player ID: 730 already e

## LADDER AND FIXTURE DATA
- also saves each json file

In [31]:
match_json_list = []

for rd in range(1, round_number + 1):
    s = httpx.get(
        "https://www.supercoach.com.au/2025/api/afl/draft/v1/leagues/13462/ladderAndFixtures?round={}&scores=true".format(rd), headers={**headers, 'User-Agent': 'Mozilla/5.0'}, verify=False)
    if s.status_code == 200:
        try:
            match_json = json.loads(s.text)
            match_json_list.append(match_json)
            with open("outputs/json files/match_json_rd{}.json".format(rd), 'w') as outfile:
                json.dump(match_json, outfile, indent=4, sort_keys=True)
            progress = (rd / round_number) * 100
            print(f"Successfully fetched and saved data for round {rd} ({progress:.2f}% complete)")
        except json.JSONDecodeError:
            print(f"Error decoding JSON for round {rd}")
    else:
        print(f"Failed to fetch data for round {rd}, status code: {s.status_code}")

Successfully fetched and saved data for round 1 (4.55% complete)
Successfully fetched and saved data for round 2 (9.09% complete)
Successfully fetched and saved data for round 3 (13.64% complete)
Successfully fetched and saved data for round 4 (18.18% complete)
Successfully fetched and saved data for round 5 (22.73% complete)
Successfully fetched and saved data for round 6 (27.27% complete)
Successfully fetched and saved data for round 7 (31.82% complete)
Successfully fetched and saved data for round 8 (36.36% complete)
Successfully fetched and saved data for round 9 (40.91% complete)
Successfully fetched and saved data for round 10 (45.45% complete)
Successfully fetched and saved data for round 11 (50.00% complete)
Successfully fetched and saved data for round 12 (54.55% complete)
Successfully fetched and saved data for round 13 (59.09% complete)
Successfully fetched and saved data for round 14 (63.64% complete)
Successfully fetched and saved data for round 15 (68.18% complete)
Succes

## COACH IDs AND NAMES

In [33]:
df_coaches = pd.DataFrame()
first_match_json = match_json_list[0]

for coach in first_match_json['ladder']:
    df_line = pd.DataFrame(
        {
            'coach_id': [coach['userTeam']['user']['id']],
            'coach_team_id': [coach['userTeam']['id']],
            'coach_first_name': [coach['userTeam']['user']['first_name']],
            'coach_team_name': [coach['userTeam']['teamname']],
        }
    )
    df_coaches = pd.concat([df_coaches, df_line])
    
df_coaches

,coach_id,coach_team_id,coach_first_name,coach_team_name
0,86078,116859,Lester,Grand Theft Zorko
0,300791,116698,Paul,SerongWayToTheTop
0,21562,116970,Mark,Sheez Nutz
0,180650,118554,Jordan,Hornehub
0,45168,116793,Luke,Duke Newcombe
0,71052,117113,James,Baker-Street Boys
0,10857,116887,Anthony,Work From Holmes
0,8258,118706,Simon,Meeky Blinders


In [34]:
df_coaches.to_csv('inputs/coach_list.csv')

## PLAYER MATCH RESULTS
Function to get player scores for each match

In [35]:
def get_player_score_data(scores, on_field_status, coach_id, coach_team_id, coach_first_name, coach_team_name):
    df_player_results = pd.DataFrame()
    df_player_line = pd.DataFrame()
    for player in scores:
        player_id = player['player_id'],
        if on_field_status == True:
            points = player['points']
        else:
            points = player['player']['player_stats'][0]['points']
        played_position = player['position']
        df_player_line = pd.DataFrame(
            {
                'round': rd_number,
                'coach_id': coach_id,
                'coach_team_id': coach_team_id,
                'coach_first_name': coach_first_name,
                'coach_team_name': coach_team_name,
                'player_id': player_id,
                'played_position': played_position,
                'points': points,
                'on_field': on_field_status
            })
        df_player_results = pd.concat([df_player_results, df_player_line])
    return df_player_results

In [36]:
df_player_match_results = pd.DataFrame()

for match in match_json_list:
    for coach in match['ladder']:
        rd_number = coach['round']
        coach_id = coach['userTeam']['user']['id'],
        coach_team_id = coach['userTeam']['id'],
        coach_first_name = coach['userTeam']['user']['first_name'],
        coach_team_name = coach['userTeam']['teamname'],
        df_on_field_players = get_player_score_data(coach['userTeam']['scores']['scoring'], True, coach_id, coach_team_id, coach_first_name, coach_team_name)
        df_off_field_players = get_player_score_data(coach['userTeam']['scores']['nonscoring'], False, coach_id, coach_team_id, coach_first_name, coach_team_name)
        df_all_players = pd.concat([df_on_field_players, df_off_field_players])
        df_player_match_results = pd.concat([df_player_match_results, df_all_players])

df_player_match_results = df_player_match_results.merge(df_player_list, on='player_id')
display(df_player_match_results)
df_player_match_results['round_number_alt'] = "R" + df_player_match_results['round_x'].astype('str')
df_player_match_results

,round_x,coach_id,coach_team_id,coach_first_name,coach_team_name,player_id,played_position,points_x,on_field,feed_id,...,ven,venavg,ven1,ven2,ven3,owned,points_per_min,total_points_per_min,mvp_value,position_change
0,1,86078,116859,Lester,Grand Theft Zorko,339,FWD,147,True,1006130,...,Kardinia Park,100.7780,"{'id': 10, 'name': 'Kardinia Park', 'display_n...","{'id': 16, 'name': 'Sydney Cricket Ground', 'd...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",95.7871,NaN,1.084956,4548.539212,None
1,1,86078,116859,Lester,Grand Theft Zorko,87,DEF,139,True,261224,...,Melbourne Cricket Ground,105.3890,"{'id': 4, 'name': 'Brisbane Cricket Ground', '...","{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 4, 'name': 'Brisbane Cricket Ground', '...",98.7464,0.000000,0.942015,4880.000000,None
2,1,86078,116859,Lester,Grand Theft Zorko,259,MID,136,True,1009199,...,Carrara Stadium,104.5510,"{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 5, 'name': 'Carrara Stadium', 'display_...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...",97.9244,1.470000,1.063188,5161.041961,None
3,1,86078,116859,Lester,Grand Theft Zorko,726,MID,119,True,1008280,...,Docklands Stadium,79.6119,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",92.0880,1.010101,1.049642,4860.391087,None
4,1,86078,116859,Lester,Grand Theft Zorko,37,MID,114,True,1017109,...,Adelaide Oval,83.0000,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",72.9346,0.728155,0.938539,5212.023618,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4043,21,180650,118554,Jordan,Hornehub,630,FWD,0,True,295467,...,Docklands Stadium,103.9700,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 17, 'name': 'ENGIE Stadium', 'display_n...",93.5060,NaN,1.049592,4867.515875,None
4044,21,180650,118554,Jordan,Hornehub,160,MID,0,False,291856,...,Melbourne Cricket Ground,107.7140,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",36.1077,NaN,1.011236,4557.777778,None
4045,21,180650,118554,Jordan,Hornehub,602,MID,0,False,998172,...,Carrara Stadium,83.8571,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 226, 'name': 'Ninja Stadium', 'display_...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",82.0797,NaN,0.945191,5146.914018,None
4046,21,180650,118554,Jordan,Hornehub,340,RUC,0,False,280317,...,Kardinia Park,81.8727,"{'id': 10, 'name': 'Kardinia Park', 'display_n...","{'id': 16, 'name': 'Sydney Cricket Ground', 'd...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",18.5162,NaN,0.701378,4910.696546,None


,round_x,coach_id,coach_team_id,coach_first_name,coach_team_name,player_id,played_position,points_x,on_field,feed_id,...,venavg,ven1,ven2,ven3,owned,points_per_min,total_points_per_min,mvp_value,position_change,round_number_alt
0,1,86078,116859,Lester,Grand Theft Zorko,339,FWD,147,True,1006130,...,100.7780,"{'id': 10, 'name': 'Kardinia Park', 'display_n...","{'id': 16, 'name': 'Sydney Cricket Ground', 'd...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",95.7871,NaN,1.084956,4548.539212,None,R1
1,1,86078,116859,Lester,Grand Theft Zorko,87,DEF,139,True,261224,...,105.3890,"{'id': 4, 'name': 'Brisbane Cricket Ground', '...","{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 4, 'name': 'Brisbane Cricket Ground', '...",98.7464,0.000000,0.942015,4880.000000,None,R1
2,1,86078,116859,Lester,Grand Theft Zorko,259,MID,136,True,1009199,...,104.5510,"{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 5, 'name': 'Carrara Stadium', 'display_...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...",97.9244,1.470000,1.063188,5161.041961,None,R1
3,1,86078,116859,Lester,Grand Theft Zorko,726,MID,119,True,1008280,...,79.6119,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",92.0880,1.010101,1.049642,4860.391087,None,R1
4,1,86078,116859,Lester,Grand Theft Zorko,37,MID,114,True,1017109,...,83.0000,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",72.9346,0.728155,0.938539,5212.023618,None,R1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4043,21,180650,118554,Jordan,Hornehub,630,FWD,0,True,295467,...,103.9700,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 17, 'name': 'ENGIE Stadium', 'display_n...",93.5060,NaN,1.049592,4867.515875,None,R21
4044,21,180650,118554,Jordan,Hornehub,160,MID,0,False,291856,...,107.7140,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",36.1077,NaN,1.011236,4557.777778,None,R21
4045,21,180650,118554,Jordan,Hornehub,602,MID,0,False,998172,...,83.8571,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 226, 'name': 'Ninja Stadium', 'display_...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",82.0797,NaN,0.945191,5146.914018,None,R21
4046,21,180650,118554,Jordan,Hornehub,340,RUC,0,False,280317,...,81.8727,"{'id': 10, 'name': 'Kardinia Park', 'display_n...","{'id': 16, 'name': 'Sydney Cricket Ground', 'd...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",18.5162,NaN,0.701378,4910.696546,None,R21


In [37]:
df_player_match_results.to_csv('outputs/player_match_results.csv')

## LADDER

In [38]:
df_ladder = pd.DataFrame()

for match in match_json_list:
    for coach in match['ladder']:
        df_line = pd.DataFrame(
            {
                'round': coach['round'],
                'coach_id': [coach['userTeam']['user']['id']],
                'coach_team_id': [coach['userTeam']['id']],
                'coach_first_name': [coach['userTeam']['user']['first_name']],
                'coach_team_name': [coach['userTeam']['teamname']],
                'wins': coach['wins'],
                'draws': coach['draws'],
                'losses': coach['losses'],
                'points_for': coach['points_for'],
                'points_against': coach['points_against'],
                'league_points': coach['points'],
                'position': coach['position']
            }
        )
        df_ladder = pd.concat([df_ladder, df_line])
df_ladder

,round,coach_id,coach_team_id,coach_first_name,coach_team_name,wins,draws,losses,points_for,points_against,league_points,position
0,1,86078,116859,Lester,Grand Theft Zorko,1,0,0,1813,1630,4,1
0,1,300791,116698,Paul,SerongWayToTheTop,1,0,0,1810,1765,4,2
0,1,21562,116970,Mark,Sheez Nutz,1,0,0,1676,1648,4,3
0,1,180650,118554,Jordan,Hornehub,1,0,0,1574,1547,4,4
0,1,45168,116793,Luke,Duke Newcombe,0,0,1,1765,1810,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...
0,21,71052,117113,James,Baker-Street Boys,10,0,11,34606,34268,40,4
0,21,10857,116887,Anthony,Work From Holmes,9,0,12,34108,34725,36,5
0,21,21562,116970,Mark,Sheez Nutz,7,0,14,35446,35564,28,6
0,21,8258,118706,Simon,Meeky Blinders,7,0,14,32517,34555,28,7


In [39]:
df_ladder.to_csv('outputs/ladder.csv')

## FIXTURE RESULTS

In [40]:
df_fixture_results = pd.DataFrame()
for match in match_json_list:
    for result in match['fixtures']:
        df_line = pd.DataFrame(
            {
                'round_number': [result['round']],
                'match_name': '{} vs {}'.format(result['user_team1']['user']['first_name'], result['user_team2']['user']['first_name']),
                'home_team_id': [result['user_team1']['id']],
                'home_team_name': [result['user_team1']['teamname']],
                'home_team_coach': [result['user_team1']['user']['first_name']],
                'home_team_points': [result['user_team1']['stats'][0]['points']],
                'away_team_id': [result['user_team2']['id']],
                'away_team_name': [result['user_team2']['teamname']],
                'away_team_coach': [result['user_team2']['user']['first_name']],
                'away_team_points': [result['user_team2']['stats'][0]['points']],
            }
        )
        df_fixture_results = pd.concat([df_fixture_results, df_line])
    
df_fixture_results

,round_number,match_name,home_team_id,home_team_name,home_team_coach,home_team_points,away_team_id,away_team_name,away_team_coach,away_team_points
0,1,Paul vs Luke,116698,SerongWayToTheTop,Paul,1810,116793,Duke Newcombe,Luke,1765
0,1,Lester vs Anthony,116859,Grand Theft Zorko,Lester,1813,116887,Work From Holmes,Anthony,1630
0,1,Mark vs James,116970,Sheez Nutz,Mark,1676,117113,Baker-Street Boys,James,1648
0,1,Jordan vs Simon,118554,Hornehub,Jordan,1574,118706,Meeky Blinders,Simon,1547
0,2,Paul vs Lester,116698,SerongWayToTheTop,Paul,1685,116859,Grand Theft Zorko,Lester,1524
...,...,...,...,...,...,...,...,...,...,...
0,20,Lester vs Mark,116859,Grand Theft Zorko,Lester,2059,116970,Sheez Nutz,Mark,1885
0,21,Paul vs Anthony,116698,SerongWayToTheTop,Paul,1360,116887,Work From Holmes,Anthony,1077
0,21,Luke vs James,116793,Duke Newcombe,Luke,1238,117113,Baker-Street Boys,James,1184
0,21,Lester vs Simon,116859,Grand Theft Zorko,Lester,1232,118706,Meeky Blinders,Simon,1169


In [41]:
df_fixture_results.to_csv('outputs/fixture_results.csv'.format(round_number))

### Rearrange fixture data to display by team + opposition scores
- Generate team rank for that week + opponent rank for that week

In [42]:
df_home_fixture_results = df_fixture_results
df_away_fixture_results = df_fixture_results

home_renamed_cols = {
     'home_team_id': 'team_id',
     'home_team_name': 'team_name',
     'home_team_coach': 'team_coach',
     'home_team_points': 'team_points',
     'away_team_id': 'opposition_team_id',
     'away_team_name': 'opposition_team_name',
     'away_team_coach': 'opposition_team_coach',
     'away_team_points': 'opposition_team_points'
}

away_renamed_cols = {
     'home_team_id': 'opposition_team_id',
     'home_team_name': 'opposition_team_name',
     'home_team_coach': 'opposition_team_coach',
     'home_team_points': 'opposition_team_points',
     'away_team_id': 'team_id',
     'away_team_name': 'team_name',
     'away_team_coach': 'team_coach',
     'away_team_points': 'team_points'
}

df_home_fixture_results = df_home_fixture_results.rename(columns=home_renamed_cols)
df_away_fixture_results = df_away_fixture_results.rename(columns=away_renamed_cols)
df_fixture_by_each_team = pd.concat([df_home_fixture_results, df_away_fixture_results])
df_fixture_by_each_team['Rank'] = df_fixture_by_each_team.groupby('round_number')['team_points'].rank(ascending=False)
df_fixture_by_each_team['Opponent Rank'] = df_fixture_by_each_team.groupby('round_number')['opposition_team_points'].rank(ascending=False)
df_fixture_by_each_team = df_fixture_by_each_team.sort_values(['round_number', 'Rank'])
df_fixture_by_each_team.to_csv('outputs/fixture_results_by_team.csv')
df_fixture_by_each_team

,round_number,match_name,team_id,team_name,team_coach,team_points,opposition_team_id,opposition_team_name,opposition_team_coach,opposition_team_points,Rank,Opponent Rank
0,1,Lester vs Anthony,116859,Grand Theft Zorko,Lester,1813,116887,Work From Holmes,Anthony,1630,1.0,6.0
0,1,Paul vs Luke,116698,SerongWayToTheTop,Paul,1810,116793,Duke Newcombe,Luke,1765,2.0,3.0
0,1,Paul vs Luke,116793,Duke Newcombe,Luke,1765,116698,SerongWayToTheTop,Paul,1810,3.0,2.0
0,1,Mark vs James,116970,Sheez Nutz,Mark,1676,117113,Baker-Street Boys,James,1648,4.0,5.0
0,1,Mark vs James,117113,Baker-Street Boys,James,1648,116970,Sheez Nutz,Mark,1676,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...
0,21,Lester vs Simon,116859,Grand Theft Zorko,Lester,1232,118706,Meeky Blinders,Simon,1169,4.0,6.0
0,21,Luke vs James,117113,Baker-Street Boys,James,1184,116793,Duke Newcombe,Luke,1238,5.0,3.0
0,21,Lester vs Simon,118706,Meeky Blinders,Simon,1169,116859,Grand Theft Zorko,Lester,1232,6.0,4.0
0,21,Mark vs Jordan,118554,Hornehub,Jordan,1154,116970,Sheez Nutz,Mark,1776,7.0,1.0


## CURRENTLY LISTED TEAMS

In [43]:
s = httpx.get("https://www.supercoach.com.au/2025/api/afl/draft/v1/leagues/13462/playersStatus", headers=headers, verify=False)
players_json = json.loads(s.text)
df_players = pd.DataFrame.from_dict(players_json, orient='index')
df_players = df_players.merge(df_player_list, on='player_id', how='left')
df_currently_listed_players = df_players.loc[df_players['trade_status'] == "owner"]
df_currently_listed_players = df_currently_listed_players.merge(df_coaches, left_on = 'user_team_id', right_on='coach_team_id', how='left')
df_currently_listed_players.to_csv('outputs/current_teams.csv')
df_currently_listed_players

,player_id,trade_status,user_team_id,waiver_until,feed_id,first_name,last_name,team,position,round,...,ven3,owned,points_per_min,total_points_per_min,mvp_value,position_change,coach_id,coach_team_id,coach_first_name,coach_team_name
0,695,owner,116970.0,NaN,297373,Marcus,Bontempelli,WBD,0,21,...,"{'id': 7, 'name': 'Docklands Stadium', 'displa...",99.42460,1.525253,1.259115,5184.863456,None,21562,116970,Mark,Sheez Nutz
1,441,owner,116698.0,NaN,290528,Max,Gawn,MEL,0,21,...,"{'id': 13, 'name': 'Melbourne Cricket Ground',...",99.69170,1.169643,1.150000,4668.528121,None,300791,116698,Paul,SerongWayToTheTop
2,529,owner,117113.0,NaN,1006121,Zak,Butters,PTA,0,21,...,"{'id': 1, 'name': 'Adelaide Oval', 'display_na...",99.19850,NaN,1.133373,5077.885237,None,71052,117113,James,Baker-Street Boys
3,75,owner,117113.0,NaN,293535,Lachie,Neale,BRL,0,21,...,"{'id': 4, 'name': 'Brisbane Cricket Ground', '...",98.66420,0.000000,0.922969,5054.158607,None,71052,117113,James,Baker-Street Boys
4,518,owner,116793.0,NaN,1004965,Tristan,Xerri,NTH,0,21,...,"{'id': 7, 'name': 'Docklands Stadium', 'displa...",98.04770,NaN,1.130268,4721.291502,None,45168,116793,Luke,Duke Newcombe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,9,owner,116698.0,NaN,1024023,Daniel,Curtin,ADE,0,21,...,"{'id': 7, 'name': 'Docklands Stadium', 'displa...",63.39910,0.750000,0.766043,6643.799472,None,300791,116698,Paul,SerongWayToTheTop
180,242,owner,116859.0,NaN,1028513,Murphy,Reid,FRE,0,21,...,"{'id': 7, 'name': 'Docklands Stadium', 'displa...",21.86600,NaN,0.692186,5292.753623,None,86078,116859,Lester,Grand Theft Zorko
181,620,owner,116887.0,NaN,1020855,Max,Hall,STK,0,21,...,"{'id': 17, 'name': 'ENGIE Stadium', 'display_n...",53.88410,NaN,0.748019,6203.179280,None,10857,116887,Anthony,Work From Holmes
182,794,owner,116970.0,NaN,1028196,Lachlan,Blakiston,ESS,0,21,...,"{'id': 13, 'name': 'Melbourne Cricket Ground',...",4.87053,0.477477,0.498084,4436.921939,None,21562,116970,Mark,Sheez Nutz


## CURRENT FREE AGENTS + WAIVER

In [44]:
df_available_players = df_players.loc[df_players['trade_status'] != "owner"]
df_available_players

,player_id,trade_status,user_team_id,waiver_until,feed_id,first_name,last_name,team,position,round,...,ven,venavg,ven1,ven2,ven3,owned,points_per_min,total_points_per_min,mvp_value,position_change
10,730,free_agent,NaN,NaN,291790,Adam,Treloar,WBD,0,21,...,Docklands Stadium,104.2540,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",58.323100,NaN,0.956811,7176.388889,None
23,125,waiver,116887.0,2025-08-03T04:00:00+10:00,1006094,Sam,Walsh,CAR,0,21,...,Perth Stadium,99.6667,"{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",81.956400,NaN,0.955823,4931.090779,None
31,7,waiver,117113.0,2025-08-03T04:00:00+10:00,297401,Matt,Crouch,ADE,0,21,...,Adelaide Oval,97.4186,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 1, 'name': 'Adelaide Oval', 'display_na...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",59.062900,NaN,1.037849,5304.416624,None
33,196,free_agent,NaN,NaN,1012013,Nic,Martin,ESS,0,21,...,Sydney Cricket Ground,76.0000,"{'id': 10, 'name': 'Kardinia Park', 'display_n...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",62.577100,NaN,0.916764,4802.046036,None
34,778,free_agent,NaN,NaN,292128,Elliot,Yeo,WCE,0,21,...,Docklands Stadium,88.3636,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 53, 'name': 'Perth Stadium', 'display_n...",35.881600,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
803,808,free_agent,NaN,NaN,1041619,Zac,Walker,WBD,0,21,...,Docklands Stadium,0.0000,"{'id': 13, 'name': 'Melbourne Cricket Ground',...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...",0.369914,NaN,NaN,NaN,None
804,766,free_agent,NaN,NaN,294859,Jeremy,*** McGovern,WCE,0,21,...,Docklands Stadium,92.8462,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 53, 'name': 'Perth Stadium', 'display_n...",37.505100,NaN,0.930435,5716.259545,None
805,101,free_agent,NaN,NaN,295518,Sam,*** Docherty,CAR,0,21,...,Perth Stadium,97.8750,"{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 13, 'name': 'Melbourne Cricket Ground',...",33.436100,NaN,0.799344,5288.759110,None
806,774,free_agent,NaN,NaN,296296,Dom,*** Sheed,WCE,0,21,...,Docklands Stadium,68.5882,"{'id': 53, 'name': 'Perth Stadium', 'display_n...","{'id': 7, 'name': 'Docklands Stadium', 'displa...","{'id': 53, 'name': 'Perth Stadium', 'display_n...",0.637074,NaN,NaN,NaN,None


In [46]:
df_available_players.to_csv('outputs/current_free_agents_waiver.csv')

# Transactions

In [47]:
s = httpx.get("https://www.supercoach.com.au/2024/api/afl/draft/v1/leagues/1283/processedWaivers", headers=headers, verify=False)
waivers_json = json.loads(s.text)
df_waiver_transactions = pd.json_normalize(waivers_json)

wanted_columns = [
    'round',
    'user_team_id',
    'processed',
    'player_id',
    'player.feed_id',
    'player.first_name',
    'player.last_name',
    'player.team.abbrev',
    'dropped_player_id',
    'dropped_player.feed_id',
    'dropped_player.first_name',
    'dropped_player.last_name',
    'dropped_player.team.abbrev',
]

df_waiver_transactions = df_waiver_transactions.loc[:, wanted_columns]
df_waiver_transactions['source'] = "Waiver"
df_waiver_transactions

,round,user_team_id,processed,player_id,player.feed_id,player.first_name,player.last_name,player.team.abbrev,dropped_player_id,dropped_player.feed_id,dropped_player.first_name,dropped_player.last_name,dropped_player.team.abbrev,source
0,2,720,2024-03-19T04:00:01+11:00,199,1015873,Archie,Perkins,ESS,181,298421,Jade,Gresham,ESS,Waiver
1,2,716,2024-03-19T04:00:01+11:00,530,280711,Charlie,Dixon,PTA,331,1004938,Gryan,Miers,GEE,Waiver
2,2,717,2024-03-19T04:00:01+11:00,434,296351,Jack,Billings,MEL,457,296420,Alex,Neal-Bullen,MEL,Waiver
3,2,719,2024-03-19T04:00:01+11:00,453,291533,Tom,McDonald,MEL,711,297899,James,Harmes,WBD,Waiver
4,2,714,2024-03-19T04:00:01+11:00,272,993799,Brayden,Fiorini,GCS,592,290627,Dion,Prestia,RIC,Waiver
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,22,720,2024-08-08T04:00:01+10:00,770,1023492,Harley,Reid,WCE,349,1008139,Toby,Bedford,GWS,Waiver
176,23,718,2024-08-13T04:00:01+10:00,682,290778,Luke,Parker,SYD,754,1023025,Reuben,Ginbey,WCE,Waiver
177,23,720,2024-08-13T04:00:01+10:00,423,998114,Jack,Scrimshaw,HAW,352,1009708,Jack,Buckley,GWS,Waiver
178,24,713,2024-08-20T04:00:01+10:00,202,1001026,Jordan,Ridley,ESS,31,1001195,Izak,Rankine,ADE,Waiver


In [ ]:
df_waiver_transactions.to_csv('outputs/waiver_transactions.csv')

## Free Agent trades

In [ ]:
s = httpx.get("https://www.supercoach.com.au/2024/api/afl/draft/v1/leagues/1283/trades", headers=headers, verify=False)
free_agent_trades_json = json.loads(s.text)
df_fa_transactions = pd.json_normalize(free_agent_trades_json)

wanted_columns = [
    'round',
    'user_id',
    'action_time',
    'buy_player.id',
    'buy_player.feed_id',
    'buy_player.first_name',
    'buy_player.last_name',
    'buy_player.team.abbrev',
    'sell_player.id',
    'sell_player.feed_id',
    'sell_player.first_name',
    'sell_player.last_name',
    'sell_player.team.abbrev',
]

df_fa_transactions = df_fa_transactions.loc[:, wanted_columns]
df_fa_transactions['source'] = "Free Agent"
df_fa_transactions

,round,user_id,action_time,buy_player.id,buy_player.feed_id,buy_player.first_name,buy_player.last_name,buy_player.team.abbrev,sell_player.id,sell_player.feed_id,sell_player.first_name,sell_player.last_name,sell_player.team.abbrev,source
0,1,713,2024-03-12T19:49:52+11:00,526,1006121,Zak,Butters,PTA,NaN,NaN,NaN,NaN,NaN,Free Agent
1,1,713,2024-03-11T19:43:17+11:00,785,1005052,Ethan,Phillips,HAW,461.0,1008541,Kysaiah,Pickett,MEL,Free Agent
2,1,713,2024-03-16T12:45:17+11:00,766,294859,Jeremy,McGovern,WCE,75.0,1002235,Cam,Rayner,BRL,Free Agent
3,1,713,2024-03-16T12:43:54+11:00,322,261510,Tom,Hawkins,GEE,461.0,1008541,Kysaiah,Pickett,MEL,Free Agent
4,1,713,2024-03-12T20:03:00+11:00,681,996765,Tom,Papley,SYD,562.0,1000223,Liam,Baker,RIC,Free Agent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
504,22,720,2024-08-06T07:05:39+10:00,480,1015781,Charlie,Comben,NTH,472.0,1017720,Jacob,van Rooyen,MEL,Free Agent
505,22,720,2024-08-08T00:36:44+10:00,204,998215,Will,Setterfield,ESS,184.0,290629,Dyson,Heppell,ESS,Free Agent
506,23,720,2024-08-13T07:07:36+10:00,355,291969,Stephen,Coniglio,GWS,679.0,993905,Callum,Mills,SYD,Free Agent
507,23,720,2024-08-13T07:07:51+10:00,302,994386,Tom,Atkins,GEE,204.0,998215,Will,Setterfield,ESS,Free Agent


In [ ]:
df_fa_transactions.to_csv('outputs/free_agent_transactions.csv')

## Team Trades

In [ ]:
s = httpx.get("https://www.supercoach.com.au/2024/api/afl/draft/v1/leagues/1283/teamtrades", headers=headers, verify=False)
team_trades_json = json.loads(s.text)

In [ ]:
df_trades_player_list = pd.DataFrame()

for trade in team_trades_json:
    df_lines = pd.json_normalize(trade['team_trade_players'])
    df_lines['round'] = trade['round']
    df_lines['user_team_id'] = trade['user_team_id']
    df_lines['processed'] = trade['action_time']
    df_lines['status'] = trade['status_id']
    df_lines['source'] = 'Trade'
    df_trades_player_list = pd.concat([df_trades_player_list,df_lines])
df_trades_player_list

,id,trade_id,player_id,trade_player_id,round,user_team_id,processed,status,source
0,30557,20885,180,68,2,713,2024-03-21T16:54:20+11:00,7,Trade
1,30558,20885,566,615,2,713,2024-03-21T16:54:20+11:00,7,Trade
0,31588,21628,372,54,2,713,2024-03-24T18:55:48+11:00,2,Trade
0,40744,27525,325,257,3,717,2024-04-01T14:58:16+11:00,7,Trade
1,40745,27525,390,0,3,717,2024-04-01T14:58:16+11:00,7,Trade
0,54401,36016,194,719,5,713,2024-04-09T13:55:21+10:00,2,Trade
1,54402,36016,577,598,5,713,2024-04-09T13:55:21+10:00,2,Trade
0,59449,39153,194,663,5,713,2024-04-11T19:26:22+10:00,7,Trade
1,59450,39153,127,154,5,713,2024-04-11T19:26:22+10:00,7,Trade
2,59451,39153,457,369,5,713,2024-04-11T19:26:22+10:00,7,Trade


In [ ]:
df_team_one_trades_player_list = df_trades_player_list.loc[df_trades_player_list['status'] == 7, ['round', 'user_team_id', 'processed', 'player_id', 'source']]
df_team_two_trades_player_list = df_trades_player_list.loc[df_trades_player_list['status'] == 7, ['round', 'user_team_id', 'processed', 'trade_player_id', 'source']]
df_team_two_trades_player_list = df_team_two_trades_player_list.rename(columns={'trade_player_id': 'player_id'})
df_trades_by_player = pd.concat([df_team_one_trades_player_list, df_team_two_trades_player_list]).sort_values(by=['processed'])
df_trades_by_player = df_trades_by_player.merge(df_player_list[['player_id', 'feed_id', 'first_name', 'last_name', 'team']], on='player_id', how='left')
df_trades_by_player = df_trades_by_player.rename(columns={'feed_id': 'player.feed_id', 'first_name': 'player.first_name', 'last_name': 'player.last_name', 'team': 'player.team.abbrev'})
df_trades_by_player

,round,user_team_id,processed,player_id,source,player.feed_id,player.first_name,player.last_name,player.team.abbrev
0,2,713,2024-03-21T16:54:20+11:00,180,Trade,1005577,Sam,Draper,ESS
1,2,713,2024-03-21T16:54:20+11:00,566,Trade,1013197,Sam,Banks,RIC
2,2,713,2024-03-21T16:54:20+11:00,615,Trade,1017959,Lance,Collard,STK
3,2,713,2024-03-21T16:54:20+11:00,68,Trade,1028520,Sam,Marshall,BRL
4,3,717,2024-04-01T14:58:16+11:00,0,Trade,NaN,NaN,NaN,NaN
5,3,717,2024-04-01T14:58:16+11:00,257,Trade,1009256,Hayden,Young,FRE
6,3,717,2024-04-01T14:58:16+11:00,390,Trade,1028566,Cody,Anderson,HAW
7,3,717,2024-04-01T14:58:16+11:00,325,Trade,1021103,Mitch,Knevitt,GEE
8,5,713,2024-04-11T19:26:22+10:00,457,Trade,1029211,Ricky,Mentha,MEL
9,5,713,2024-04-11T19:26:22+10:00,372,Trade,1032119,James,Leake,GWS


## Combined Free Agents, Waiver, and Trades transactions list

In [ ]:
columns = [
    'round',
    'user_team_id',
    'processed',
    'player_id',
    'player.feed_id',
    'player.first_name',
    'player.last_name',
    'player.team.abbrev',
    'dropped_player_id',
    'dropped_player.feed_id',
    'dropped_player.first_name',
    'dropped_player.last_name',
    'dropped_player.team.abbrev',
    'source',
]
df_waiver_transactions.columns = columns
df_fa_transactions.columns = columns

df_transactions = pd.concat([df_waiver_transactions, df_fa_transactions, df_trades_by_player])
df_transactions = df_transactions.sort_values(by=['processed'])
df_transactions

,round,user_team_id,processed,player_id,player.feed_id,player.first_name,player.last_name,player.team.abbrev,dropped_player_id,dropped_player.feed_id,dropped_player.first_name,dropped_player.last_name,dropped_player.team.abbrev,source
149,1,715,2024-02-22T20:30:03+11:00,110,1001398,Matthew,Kennedy,CAR,NaN,NaN,NaN,NaN,NaN,Free Agent
148,1,715,2024-02-22T20:30:03+11:00,116,1023446,Alex,Mirkov,CAR,NaN,NaN,NaN,NaN,NaN,Free Agent
159,1,715,2024-02-22T20:30:06+11:00,110,1001398,Matthew,Kennedy,CAR,NaN,NaN,NaN,NaN,NaN,Free Agent
158,1,715,2024-02-22T20:30:06+11:00,766,294859,Jeremy,McGovern,WCE,NaN,NaN,NaN,NaN,NaN,Free Agent
157,1,715,2024-02-22T20:30:06+11:00,686,1006126,James,Rowbottom,SYD,NaN,NaN,NaN,NaN,NaN,Free Agent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,24,713,2024-08-21T11:08:52+10:00,124,295461,Adam,Saad,CAR,766.0,294859,Jeremy,McGovern,WCE,Free Agent
95,24,713,2024-08-23T15:22:04+10:00,340,997230,Tyson,Stengle,GEE,99.0,996731,Charlie,Curnow,CAR,Free Agent
96,24,713,2024-08-24T13:14:14+10:00,311,1017063,Toby,Conway,GEE,339.0,280317,Rhys,Stanley,GEE,Free Agent
97,24,713,2024-08-25T05:19:45+10:00,521,1027870,Tom,Anastasopoulos,PTA,704.0,1017126,Sam,Darcy,WBD,Free Agent


In [ ]:
df_transactions.to_csv('outputs/transactions.csv')

## Add source to player match results

In [13]:
def get_transaction_status(row):
    post_transaction_list = pd.DataFrame()
    round_number = row['round']
    feed_id = row['player.feed_id']
    source = row['source']
    player_rows = df_player_match_results.loc[(df_player_match_results['round_x'] >= round_number) & (df_player_match_results['feed_id'] == feed_id), :]
    player_rows['source'] = source
    if len(player_rows) > 0:
        df_player_match_results.update(player_rows)

df_player_match_results['source'] = "Draft"
df_transactions.apply(lambda row: get_transaction_status(row), axis=1)
df_player_match_results

NameError: name 'df_player_match_results' is not defined

In [14]:
df_player_match_results.to_csv('outputs/player_match_results.csv')

NameError: name 'df_player_match_results' is not defined

### Create folder and save all outputs

In [15]:
import os
import time

timestr = time.strftime("%Y%m%d-%H%M%S")
destination = "exports/supercoach_league_scrape_{}_R{}_{}".format(year_number, round_number, timestr)

if not os.path.exists(destination):
    os.makedirs(destination)

df_coaches.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/coach_list.csv".format(year_number, round_number, timestr)) 
df_player_match_results.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/player_match_results.csv".format(year_number, round_number, timestr))
df_ladder.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/ladder.csv".format(year_number, round_number, timestr))
df_fixture_results.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/fixture_results.csv".format(year_number, round_number, timestr))
df_fixture_by_each_team.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/fixture_results_by_team.csv".format(year_number, round_number, timestr))
df_currently_listed_players.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/current_teams.csv".format(year_number, round_number, timestr))
df_available_players.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/current_free_agents_waiver.csv".format(year_number, round_number, timestr))
df_waiver_transactions.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/waiver_transactions.csv".format(year_number, round_number, timestr))
df_fa_transactions.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/free_agent_transactions.csv".format(year_number, round_number, timestr))
df_transactions.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/transactions.csv".format(year_number, round_number, timestr))
df_player_match_results.to_csv("exports/supercoach_league_scrape_{}_R{}_{}/player_match_results.csv".format(year_number, round_number, timestr))

NameError: name 'df_player_match_results' is not defined